# MorphoNet-BT: Final Five-Fold Held-Out Evaluation

Architecture v1.0 • Phase 7 • Patient-separated outer cross-validation

This notebook evaluates the five validation-selected best checkpoints on their corresponding held-out test folds. Slice-level performance is the primary pooled analysis; mean-probability patient-level performance is reported separately.

## Public-copy execution policy
This is an English rendering of a historical research notebook. Outputs and execution counts were cleared; no cell was rerun for Phase 18. Historical findings below describe earlier runs only. Every code cell is gated, including setup cells that may perform model checks, preprocessing or writes. Enable `CONFIRM_PUBLIC_WORKFLOW` deliberately in the first code cell only after reviewing [the reproduction guide](../docs/reproducibility.md). Existing stage-specific confirmations must also be enabled separately. Run from the repository root or set `MORPHONET_ROOT` to a private writable experiment workspace with the same source/config layout; set `MORPHONET_DATA_ROOT` for dataset discovery. Original source/configuration fingerprints, private split manifests and checkpoints are not portable artifacts; do not bypass their validation. See the guide for prerequisites and limitations.

**Model identity:** this phase concerns the original full model, later A5. The principal manuscript A4 uses `build_ablation("A4")` and fixed mean fusion, not AMGF.


## Evaluation protocol

Every MRI slice and every patient belongs to exactly one outer test fold. No test result is used for model selection, retraining, augmentation changes, or tuning. Only `best_macro_f1.pt` is loaded. No optimizer or scheduler is constructed. The model receives MRI tensors only.

Test preprocessing is the approved deterministic grayscale, bilinear resize to 256 × 256, and per-image Z-score without clipping. The frozen configuration retains training augmentation settings, but explicit `mode="test"` disables every transformation. Ground-truth morphology is not returned by the test dataset or supplied to the model.

Each fold runs one inference pass. Completed predictions are protected by SHA-256 receipts and reused on notebook reruns. An interrupted, unreceipted inference stops for review instead of automatically repeating the test.

## Imports and reproducibility

Inference uses float32, deterministic algorithms, a fixed seed of 42, and disabled TF32. Software and device details are saved alongside the results. The numerical inference policy is identical for all folds and fixed before inspecting test performance.

In [ ]:
CONFIRM_PUBLIC_WORKFLOW = False  # Explicit opt-in; no cell has been rerun.
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.evaluation.protocol import preflight, evaluate_fold, audit_oof, sha256, save_json, require
from src.evaluation.analysis import metrics, save_analysis, METRICS
from src.evaluation.figures import save_figures

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 4)
OUTPUT = ROOT / "results/main"

## Architecture, checkpoint, and test-split verification

All five checkpoints are checked before any test inference: completion markers, confirmed best epochs and validation scores, model configuration, architecture fingerprint, preprocessing source hashes, and all frozen split hashes. Strict state-dictionary loading checks compatibility. Any mismatch aborts evaluation.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
context = preflight(ROOT)
display(context["checkpoints"][["Fold", "BestEpoch", "ValidationMacroF1", "TestPatients", "TestSlices"]])
display(pd.DataFrame(context["metadata"]["software"].items(), columns=["Component", "Version"]))
print("Device:", context["metadata"]["device"])

## Patient separation

The three partitions in each fold must have disjoint PID sets. The outer test folds must cover all 3,064 samples and 233 patients exactly once. These checks use the original manifest and the frozen split CSVs, before reading test images.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(context["split_audit"])
print("Outer split coverage: 3,064 samples and 233 patients; each appears in one test fold.")
fold_predictions = {}

## Evaluation functions and metric definitions

Reusable code is maintained in `src/evaluation/`. Predictions use the softmax probabilities in the fixed order Meningioma, Glioma, Pituitary. Labels use argmax (ties resolve to the lowest class index). Specificity is TN/(TN+FP), and class AUC is one-vs-rest. Macro AUC is the arithmetic mean of the three class AUCs. Undefined metrics are stored as NaN and propagate into macro summaries; they are never silently omitted.

Fold-level metrics describe each held-out partition. Aggregated OOF metrics are recomputed from concatenated predictions, not averaged confusion matrices.

## Fold 1: held-out inference

Load the verified validation-selected checkpoint and evaluate its held-out test partition once. The compact output confirms sample and patient counts; all probabilities are saved to CSV.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
fold_predictions[1] = evaluate_fold(context, 1)

## Fold 2: held-out inference

Load the verified validation-selected checkpoint and evaluate its held-out test partition once. The compact output confirms sample and patient counts; all probabilities are saved to CSV.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
fold_predictions[2] = evaluate_fold(context, 2)

## Fold 3: held-out inference

Load the verified validation-selected checkpoint and evaluate its held-out test partition once. The compact output confirms sample and patient counts; all probabilities are saved to CSV.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
fold_predictions[3] = evaluate_fold(context, 3)

## Fold 4: held-out inference

Load the verified validation-selected checkpoint and evaluate its held-out test partition once. The compact output confirms sample and patient counts; all probabilities are saved to CSV.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
fold_predictions[4] = evaluate_fold(context, 4)

## Fold 5: held-out inference

Load the verified validation-selected checkpoint and evaluate its held-out test partition once. The compact output confirms sample and patient counts; all probabilities are saved to CSV.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
fold_predictions[5] = evaluate_fold(context, 5)

## OOF integrity gate

Final metrics are computed only after verifying exact identities, true labels, probability validity, source paths, and fold assignment against each frozen test CSV. Duplicate, missing, unexpected, or cross-fold patient records cause an immediate stop.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
oof, integrity = audit_oof(context, fold_predictions)
display(pd.DataFrame(integrity.items(), columns=["Integrity check", "Observed"]))
results = save_analysis(context, oof, fold_predictions)
tables = results["tables"]

## Five-fold held-out results

Each row reports one independently held-out partition. Fold summaries use sample standard deviation (ddof = 1). The folds share overlapping training populations, so these five values are descriptive variability estimates rather than independent replications.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables["five_fold_test_results"])

## Aggregated slice-level performance

Pooled OOF estimates use all 3,064 predictions. They differ from equal-weight fold means because sample counts and nonlinear metric definitions differ. Values are proportions, not percentages.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables["final_performance_summary"])

## Per-class slice-level metrics

Sensitivity is recall. Support gives the number of true-class slices; predicted support helps identify undefined precision. AUC is calculated from continuous softmax scores, not discrete predicted labels.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables["oof_per_class_metrics"])
undefined = tables["oof_per_class_metrics"].select_dtypes("number").isna().sum().sum()
print(f"Undefined aggregated slice-level metric values: {undefined}")

## Aggregated confusion matrix and ROC analysis

Rows are true classes and columns are predicted classes. The primary matrix contains raw counts; the secondary matrix normalizes each true-class row to percentages. ROC curves show the three one-vs-rest tradeoffs, with class AUCs and their macro mean.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
figure_paths = save_figures(ROOT, oof, results)
display(results["slice_matrix"])
for name in ["oof_confusion_matrix", "oof_confusion_matrix_normalized", "oof_roc_curves"]:
    display(Image(filename=str(OUTPUT / "figures" / f"{name}.png"), width=540))

## Secondary patient-level aggregation

For each PID, average all slice softmax probabilities and take argmax. Patient truth and test-fold membership must each be constant within PID. Every patient contributes one decision regardless of slice count. This analysis uses the same saved predictions and performs no additional model inference.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
patients = tables["patient_level_predictions"]
print(f"Patient predictions: {len(patients)}")
display(patients.head(5))

## Patient-level metrics and confusion matrix

These metrics quantify 233 patient decisions and must not be mixed with slice-level results. Both the per-class denominators and confusion counts now refer to patients.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables["patient_level_metrics"])
display(tables["patient_level_per_class_metrics"])
display(results["patient_matrix"])
display(Image(filename=str(OUTPUT / "figures/patient_confusion_matrix.png"), width=540))

## Patient-cluster bootstrap: 95% confidence interval

Resample 233 PIDs uniformly with replacement for each of 2,000 iterations (seed 42). Include every slice of each sampled PID, including multiplicities when a PID is selected repeatedly. The replicate accuracy is total correct slices divided by total sampled slices, so denominators can vary. The 2.5th and 97.5th percentiles form the interval.

This procedure preserves within-patient clustering. It conditions on the fixed OOF predictions and does not capture uncertainty from retraining, checkpoint selection, or a new cross-validation partition. It is not a claim of independent slices or external clinical validation.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables["accuracy_cluster_bootstrap_ci"])

## Fold variability and descriptive Fold 2 analysis

Fold 2 follows exactly the same evaluation policy. The following class precision, recall, support, confusion counts, and patient metrics describe its errors after evaluation. They do not authorize changing the frozen experiment.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
fold2_summary, fold2_classes, fold2_matrix = metrics(fold_predictions[2])
display(fold2_classes)
display(fold2_matrix)
display(tables["patient_fold_metrics"])
display(tables["patient_fold_per_class_metrics"].query("Fold == 2"))
for name in ["fold_macro_f1", "fold_accuracy"]:
    display(Image(filename=str(OUTPUT / "figures" / f"{name}.png"), width=540))

## Misclassification and patient consistency

All difficult cases remain in the analysis. Confusion pairs summarize directional errors. The consistency file contains each patient's slice count, correct count, within-patient slice accuracy, mean probabilities, and patient prediction. A patient with some incorrect slices can still have a correct mean-probability decision.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
print(f"Misclassified slices: {len(tables['misclassified_samples'])} / {len(oof)}")
print(f"Misclassified patient decisions: {len(tables['misclassified_patients'])} / {len(patients)}")
display(tables["misclassification_pairs"])
display(tables["patient_consistency"].sort_values("slice_accuracy").head(8))

## Publication tables, figures, and reproducibility metadata

All full tables are saved under `results/main/`; fold predictions and confusion matrices are in the corresponding fold directories. Six figures are saved as 400-dpi PNG and vector PDF. Metadata records the architecture, preprocessing, software, device, checkpoint hashes, and bootstrap protocol. Existing training records and model checkpoints are preserved.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
metadata = context["metadata"]
metadata["oof_integrity"] = integrity
metadata["evaluation_sources_sha256"] = {
    str(path.relative_to(ROOT)): sha256(path)
    for path in sorted((ROOT / "src/evaluation").glob("*.py"))
}
metadata["receipts"] = {
    str(fold): json.loads((OUTPUT / f"fold_{fold}/test_evaluation_receipt.json").read_text())
    for fold in range(1, 6)
}
metadata["figures"] = [str(Path(path).relative_to(ROOT)) for path in figure_paths]
save_json(OUTPUT / "evaluation_metadata.json", metadata)
artifacts = sorted(OUTPUT.glob("*.csv")) + [Path(path) for path in figure_paths]
display(pd.DataFrame({"Artifact group": ["Root CSV tables", "Publication figures", "Fold predictions"],
                      "Files": [len(list(OUTPUT.glob("*.csv"))), len(figure_paths), 5]}))

## Final scientific audit

Checkpoint and training-completion hashes are compared with their pre-inference values. Frozen model, preprocessing, and split checks are repeated without evaluating images. The external execution runner additionally verifies that all notebook source and text outputs are English only and contain no error outputs.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
for row in context["checkpoints"].itertuples(index=False):
    require(sha256(row.Checkpoint) == row.CheckpointSHA256, "Checkpoint modified")
    require(sha256(OUTPUT / f"fold_{row.Fold}/training_complete.json") == row.CompletionSHA256,
            "Training completion marker modified")
verified = preflight(ROOT)
checks = {
    "Five best checkpoints only": len(metadata["receipts"]) == 5,
    "One inference pass per fold": all(r["inference_passes"] == 1 for r in metadata["receipts"].values()),
    "3064 unique OOF samples": len(oof) == oof.sample_id.nunique() == 3064,
    "233 patients in exactly one test fold": oof.PID.nunique() == 233 and oof.groupby("PID").fold.nunique().eq(1).all(),
    "Deterministic test mode; MRI-only model input": metadata["augmentation"] is False,
    "Patient-cluster bootstrap": metadata["bootstrap"]["unit"] == "patient",
    "Six PNG and six PDF figures": len(figure_paths) == 12 and all(Path(p).is_file() for p in figure_paths),
    "Patient and slice metrics saved": (OUTPUT / "patient_level_metrics.csv").is_file() and (OUTPUT / "final_performance_summary.csv").is_file(),
}
checks = {name: bool(value) for name, value in checks.items()}
require(all(checks.values()), "Final scientific audit failed")
save_json(OUTPUT / "final_scientific_audit.json", checks)
display(pd.DataFrame({"Check": checks.keys(), "Status": ["PASS" if value else "FAIL" for value in checks.values()]}))

## Final conclusions

The following summary is generated from fixed saved predictions. Fold variability and class-specific weaknesses should remain visible in subsequent baseline comparisons. No baseline or ablation experiment is started here.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
summary = results["slice_summary"]
patient_summary = results["patient_summary"]
ci = tables["accuracy_cluster_bootstrap_ci"].iloc[0]
folds = tables["five_fold_test_results"]
errors = fold2_classes.assign(Errors=fold2_matrix.sum(axis=1).to_numpy()-np.diag(fold2_matrix))
worst = errors.loc[errors.Errors.idxmax()]
display(Markdown(
    f"OOF slice accuracy is **{summary['Accuracy']:.4%}** "
    f"(patient-cluster 95% CI: **{ci.Lower95:.4%}–{ci.Upper95:.4%}**), "
    f"with macro F1 **{summary['Macro F1']:.4f}** and macro AUC **{summary['Macro ROC-AUC']:.4f}**. "
    f"Patient accuracy is **{patient_summary['Accuracy']:.4%}**, macro F1 **{patient_summary['Macro F1']:.4f}**, "
    f"and macro AUC **{patient_summary['Macro ROC-AUC']:.4f}**.\n\n"
    f"Held-out fold macro F1 ranges from **{folds['Macro F1'].min():.4f}** to **{folds['Macro F1'].max():.4f}**. "
    f"Fold 2 has macro F1 **{fold2_summary['Macro F1']:.4f}**; its largest true-class error count is "
    f"**{worst['Class']} ({int(worst.Errors)} slices)**. "
    "These results support a transparent baseline comparison under the same frozen splits, while fold variability "
    "must be reported and does not establish external clinical generalization.\n\n"
    "**Phase 7 ends here. Baseline training and ablation studies require separate approval.**"
))